In [1]:
# ==========================================================
# STEP 1 : Project Initialization
# ==========================================================

from pathlib import Path
import os

# ----------------------------------------------------------
# Project Root
# ----------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

# ----------------------------------------------------------
# Project Directories
# ----------------------------------------------------------

DATASETS_DIR = PROJECT_ROOT / "datasets"

RAW_DATASET_DIR = DATASETS_DIR / "raw"

MASTER_DATASET_DIR = DATASETS_DIR / "master_skin_dataset"

BALANCED_DATASET_DIR = DATASETS_DIR / "balanced_skin_dataset"

FINAL_SPLIT_DATASET_DIR = DATASETS_DIR / "final_split_dataset"

KAGGLE_DIR = PROJECT_ROOT / "kaggle"

MODELS_DIR = PROJECT_ROOT / "models"

OUTPUTS_DIR = PROJECT_ROOT / "outputs"

LOGS_DIR = PROJECT_ROOT / "logs"

# ----------------------------------------------------------
# Verify Project Structure
# ----------------------------------------------------------

directories = [
    RAW_DATASET_DIR,
    MASTER_DATASET_DIR,
    BALANCED_DATASET_DIR,
    FINAL_SPLIT_DATASET_DIR,
    KAGGLE_DIR,
    MODELS_DIR,
    OUTPUTS_DIR,
    LOGS_DIR,
]

print("=" * 60)
print("PROJECT INITIALIZATION")
print("=" * 60)

print(f"Project Root            : {PROJECT_ROOT}")
print()

print("Project Directories")

for directory in directories:
    status = "✓ Exists" if directory.exists() else "✗ Missing"
    print(f"{status:<10} {directory.relative_to(PROJECT_ROOT)}")

print("\nProject initialization completed successfully.")
print("=" * 60)

PROJECT INITIALIZATION
Project Root            : C:\Users\vnman\Desktop\virtual-internship

Project Directories
✓ Exists   datasets\raw
✓ Exists   datasets\master_skin_dataset
✓ Exists   datasets\balanced_skin_dataset
✓ Exists   datasets\final_split_dataset
✓ Exists   kaggle
✓ Exists   models
✓ Exists   outputs
✓ Exists   logs

Project initialization completed successfully.


In [2]:
# ==========================================================
# STEP 2 : Configure Kaggle API
# ==========================================================

import os

# ----------------------------------------------------------
# Configure Kaggle
# ----------------------------------------------------------

KAGGLE_CONFIG_DIR = KAGGLE_DIR

os.environ["KAGGLE_CONFIG_DIR"] = str(KAGGLE_CONFIG_DIR)

print("=" * 60)
print("KAGGLE CONFIGURATION")
print("=" * 60)

print(f"Kaggle Config Directory : {KAGGLE_CONFIG_DIR}")

if (KAGGLE_CONFIG_DIR / "kaggle.json").exists():
    print("✓ kaggle.json found")
else:
    raise FileNotFoundError("kaggle.json not found inside the kaggle folder.")

print("\nKaggle API configured successfully.")
print("=" * 60)

KAGGLE CONFIGURATION
Kaggle Config Directory : C:\Users\vnman\Desktop\virtual-internship\kaggle
✓ kaggle.json found

Kaggle API configured successfully.


In [3]:
# ==========================================================
# STEP 3 : Verify Kaggle Connection
# ==========================================================

import subprocess

print("=" * 60)
print("VERIFYING KAGGLE CONNECTION")
print("=" * 60)

try:
    result = subprocess.run(
        ["kaggle", "datasets", "list", "-s", "skin"],
        capture_output=True,
        text=True,
        check=True
    )

    print("✓ Successfully connected to Kaggle.\n")

    print("Top matching datasets:\n")
    print(result.stdout[:1000])   # Display first part of the output

except subprocess.CalledProcessError as e:
    print("✗ Failed to connect to Kaggle.\n")
    print(e.stderr)

print("=" * 60)

VERIFYING KAGGLE CONNECTION
✓ Successfully connected to Kaggle.

Top matching datasets:

ref                                                                title                                                 size  lastUpdated                 downloadCount  voteCount  usabilityRating  
-----------------------------------------------------------------  ----------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
kmader/skin-cancer-mnist-ham10000                                  Skin Cancer MNIST: HAM10000                     5582914511  2018-09-20 20:36:13.037000         277470       2376  0.7058824        
surajghuwalewala/ham1000-segmentation-and-classification           Skin cancer: HAM10000                           2781385274  2021-05-27 09:08:46.813000          20949        154  0.9411765        
nodoubttome/skin-cancer9-classesisic                               Skin Cancer ISIC                                

In [4]:
# ==========================================================
# STEP 4 : Download Dataset 1
# Oily, Dry and Normal Skin Types
# ==========================================================

import subprocess

DATASET_NAME = "shakyadissanayake/oily-dry-and-normal-skin-types-dataset"

print("=" * 60)
print("DOWNLOADING DATASET 1")
print("=" * 60)

print(f"Dataset : {DATASET_NAME}")
print(f"Destination : {RAW_DATASET_DIR}")

subprocess.run([
    "kaggle",
    "datasets",
    "download",
    "-d",
    DATASET_NAME,
    "-p",
    str(RAW_DATASET_DIR)
], check=True)

print("\n✓ Download completed successfully.")

zip_file = RAW_DATASET_DIR / "oily-dry-and-normal-skin-types-dataset.zip"

if zip_file.exists():
    print(f"\nDownloaded File : {zip_file.name}")
    print(f"Location        : {zip_file}")
else:
    print("\nDownload finished but zip file was not found.")

print("=" * 60)

DOWNLOADING DATASET 1
Dataset : shakyadissanayake/oily-dry-and-normal-skin-types-dataset
Destination : C:\Users\vnman\Desktop\virtual-internship\datasets\raw

✓ Download completed successfully.

Downloaded File : oily-dry-and-normal-skin-types-dataset.zip
Location        : C:\Users\vnman\Desktop\virtual-internship\datasets\raw\oily-dry-and-normal-skin-types-dataset.zip


In [5]:
# ==========================================================
# STEP 5 : Extract Dataset 1
# Oily, Dry and Normal Skin Types
# ==========================================================

import zipfile

ZIP_FILE = RAW_DATASET_DIR / "oily-dry-and-normal-skin-types-dataset.zip"

EXTRACT_DIR = RAW_DATASET_DIR / "dataset_normal"

EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 60)
print("EXTRACTING DATASET 1")
print("=" * 60)

print(f"ZIP File         : {ZIP_FILE.name}")
print(f"Extract Location : {EXTRACT_DIR}")

with zipfile.ZipFile(ZIP_FILE, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

print("\n✓ Extraction completed successfully.")

print("\nTop-level contents:")

for item in sorted(EXTRACT_DIR.iterdir()):
    print(f"• {item.name}")

print("=" * 60)

EXTRACTING DATASET 1
ZIP File         : oily-dry-and-normal-skin-types-dataset.zip
Extract Location : C:\Users\vnman\Desktop\virtual-internship\datasets\raw\dataset_normal

✓ Extraction completed successfully.

Top-level contents:
• Oily-Dry-Skin-Types


In [6]:
# ============================================================
# STEP 6 : Consolidate Normal Images into Master Dataset
# ============================================================

import shutil

print("=" * 60)
print("CONSOLIDATING NORMAL IMAGES")
print("=" * 60)

# Destination folder
normal_dest = MASTER_DATASET_DIR / "Normal"
normal_dest.mkdir(parents=True, exist_ok=True)

# Source folders (same as original Colab notebook)
source_normal_folders = [
    RAW_DATASET_DIR / "dataset_normal" / "Oily-Dry-Skin-Types" / "train" / "normal",
    RAW_DATASET_DIR / "dataset_normal" / "Oily-Dry-Skin-Types" / "test" / "normal",
    RAW_DATASET_DIR / "dataset_normal" / "Oily-Dry-Skin-Types" / "valid" / "normal"
]

counter = 0

for folder in source_normal_folders:
    if folder.exists():
        print(f"\nProcessing : {folder.relative_to(PROJECT_ROOT)}")

        for file_path in folder.glob("*"):
            if file_path.suffix.lower() in [".png", ".jpg", ".jpeg"]:
                new_filename = f"normal_{counter}{file_path.suffix}"
                shutil.copy(file_path, normal_dest / new_filename)
                counter += 1

# Remove temporary extraction folder (same logic as original)
temp_extract_folder = RAW_DATASET_DIR / "dataset_normal"

if temp_extract_folder.exists():
    shutil.rmtree(temp_extract_folder)
    print("\n✓ Temporary extraction folder removed.")

print("\n✓ Consolidation completed successfully.\n")

print("Summary")
print(f"Destination   : {normal_dest}")
print(f"Images Copied : {counter}")

print("=" * 60)

CONSOLIDATING NORMAL IMAGES

Processing : datasets\raw\dataset_normal\Oily-Dry-Skin-Types\train\normal

Processing : datasets\raw\dataset_normal\Oily-Dry-Skin-Types\test\normal

Processing : datasets\raw\dataset_normal\Oily-Dry-Skin-Types\valid\normal

✓ Temporary extraction folder removed.

✓ Consolidation completed successfully.

Summary
Destination   : C:\Users\vnman\Desktop\virtual-internship\datasets\master_skin_dataset\Normal
Images Copied : 1274


In [7]:
# ============================================================
# STEP 7 : Download Dataset 2
# ============================================================

import os
import subprocess
import zipfile

print("=" * 60)
print("DOWNLOADING DATASET 2")
print("=" * 60)

dataset_name = "trainingdatapro/skin-defects-acne-redness-and-bags-under-the-eyes"

zip_file = RAW_DATASET_DIR / "skin-defects-acne-redness-and-bags-under-the-eyes.zip"
extract_path_redness = RAW_DATASET_DIR / "dataset_redness"

extract_path_redness.mkdir(parents=True, exist_ok=True)

print(f"Dataset           : {dataset_name}")
print(f"Destination       : {RAW_DATASET_DIR}")

# Download dataset
result = subprocess.run(
    [
        "kaggle",
        "datasets",
        "download",
        "-d",
        dataset_name,
        "-p",
        str(RAW_DATASET_DIR),
    ],
    capture_output=True,
    text=True,
)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Dataset download failed.")

print("\n✓ Download completed successfully.")

print(f"\nDownloaded File : {zip_file.name}")

print("\nExtracting dataset...")

with zipfile.ZipFile(zip_file, "r") as zip_ref:
    zip_ref.extractall(extract_path_redness)

print("✓ Extraction completed successfully.")

# Remove downloaded zip (same logic as original notebook)
if zip_file.exists():
    zip_file.unlink()

print("✓ ZIP file removed.")

print("\nTop-level contents:")

for item in extract_path_redness.iterdir():
    print(f"• {item.name}")

print("\nFolder Structure Found")

for root, dirs, files in os.walk(extract_path_redness):
    if files:
        image_files = [
            f for f in files
            if f.lower().endswith((".png", ".jpg", ".jpeg"))
        ]

        relative_path = os.path.relpath(root, extract_path_redness)

        print(f"{relative_path} | Images found : {len(image_files)}")

print("\nSummary")
print(f"Extract Location : {extract_path_redness}")

print("=" * 60)

DOWNLOADING DATASET 2
Dataset           : trainingdatapro/skin-defects-acne-redness-and-bags-under-the-eyes
Destination       : C:\Users\vnman\Desktop\virtual-internship\datasets\raw

✓ Download completed successfully.

Downloaded File : skin-defects-acne-redness-and-bags-under-the-eyes.zip

Extracting dataset...
✓ Extraction completed successfully.
✓ ZIP file removed.

Top-level contents:
• files
• skin_defects.csv

Folder Structure Found
. | Images found : 0
files\acne\0 | Images found : 3
files\acne\1 | Images found : 3
files\acne\2 | Images found : 3
files\acne\3 | Images found : 3
files\acne\4 | Images found : 3
files\acne\5 | Images found : 3
files\acne\6 | Images found : 3
files\acne\7 | Images found : 3
files\acne\8 | Images found : 3
files\acne\9 | Images found : 3
files\bags\10 | Images found : 3
files\bags\11 | Images found : 3
files\bags\12 | Images found : 3
files\bags\13 | Images found : 3
files\bags\14 | Images found : 3
files\bags\15 | Images found : 3
files\bags\16 | I

In [8]:
# ============================================================
# STEP 8 : Consolidate Redness Images into Master Dataset
# ============================================================

import shutil

print("=" * 60)
print("CONSOLIDATING REDNESS IMAGES")
print("=" * 60)

# Destination folder
redness_dest = MASTER_DATASET_DIR / "Redness"
redness_dest.mkdir(parents=True, exist_ok=True)

# Source folder (same logic as original notebook)
source_redness_root = (
    RAW_DATASET_DIR
    / "dataset_redness"
    / "files"
    / "redness"
)

counter = 0

if source_redness_root.exists():

    print(f"Processing : {source_redness_root.relative_to(PROJECT_ROOT)}")

    # Original notebook uses rglob("*")
    for file_path in source_redness_root.rglob("*"):

        if (
            file_path.is_file()
            and file_path.suffix.lower() in [".png", ".jpg", ".jpeg"]
        ):

            new_filename = f"redness_{counter}{file_path.suffix}"

            shutil.copy(file_path, redness_dest / new_filename)

            counter += 1

# Remove temporary extraction folder (same as original)
temp_extract_folder = RAW_DATASET_DIR / "dataset_redness"

if temp_extract_folder.exists():
    shutil.rmtree(temp_extract_folder)
    print("\n✓ Temporary extraction folder removed.")

print("\n✓ Consolidation completed successfully.\n")

print("Summary")
print(f"Destination   : {redness_dest}")
print(f"Images Copied : {counter}")

print("=" * 60)

CONSOLIDATING REDNESS IMAGES
Processing : datasets\raw\dataset_redness\files\redness

✓ Temporary extraction folder removed.

✓ Consolidation completed successfully.

Summary
Destination   : C:\Users\vnman\Desktop\virtual-internship\datasets\master_skin_dataset\Redness
Images Copied : 30


In [9]:
# ============================================================
# STEP 9 : Download Dataset 3
# ============================================================

import os
import subprocess
import zipfile

print("=" * 60)
print("DOWNLOADING DATASET 3")
print("=" * 60)

dataset_name = "ahmedismaiil/skin-issues-version-2-dataset-balanced"

zip_file = RAW_DATASET_DIR / "skin-issues-version-2-dataset-balanced.zip"
extract_path_issues = RAW_DATASET_DIR / "dataset_issues"

extract_path_issues.mkdir(parents=True, exist_ok=True)

print(f"Dataset           : {dataset_name}")
print(f"Destination       : {RAW_DATASET_DIR}")

# Download dataset
result = subprocess.run(
    [
        "kaggle",
        "datasets",
        "download",
        "-d",
        dataset_name,
        "-p",
        str(RAW_DATASET_DIR),
    ],
    capture_output=True,
    text=True,
)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Dataset download failed.")

print("\n✓ Download completed successfully.")

print(f"\nDownloaded File : {zip_file.name}")

print("\nExtracting dataset...")

with zipfile.ZipFile(zip_file, "r") as zip_ref:
    zip_ref.extractall(extract_path_issues)

print("✓ Extraction completed successfully.")

# Remove downloaded ZIP (same logic as original notebook)
if zip_file.exists():
    zip_file.unlink()

print("✓ ZIP file removed.")

print("\nTop-level contents:")

for item in extract_path_issues.iterdir():
    print(f"• {item.name}")

print("\nFolder Structure Found")

for root, dirs, files in os.walk(extract_path_issues):
    if files:
        image_files = [
            f for f in files
            if f.lower().endswith((".png", ".jpg", ".jpeg"))
        ]

        relative_path = os.path.relpath(root, extract_path_issues)

        print(f"{relative_path} | Images found : {len(image_files)}")

print("\nSummary")
print(f"Extract Location : {extract_path_issues}")

print("=" * 60)

DOWNLOADING DATASET 3
Dataset           : ahmedismaiil/skin-issues-version-2-dataset-balanced
Destination       : C:\Users\vnman\Desktop\virtual-internship\datasets\raw

✓ Download completed successfully.

Downloaded File : skin-issues-version-2-dataset-balanced.zip

Extracting dataset...
✓ Extraction completed successfully.
✓ ZIP file removed.

Top-level contents:
• Skin v2

Folder Structure Found
Skin v2\acne | Images found : 2060
Skin v2\blackheades | Images found : 1970
Skin v2\dark spots | Images found : 2126
Skin v2\pores | Images found : 1632
Skin v2\wrinkles | Images found : 1982

Summary
Extract Location : C:\Users\vnman\Desktop\virtual-internship\datasets\raw\dataset_issues


In [10]:
# ============================================================
# STEP 10 : Consolidate Skin Issues into Master Dataset
# ============================================================

import shutil

print("=" * 60)
print("CONSOLIDATING SKIN ISSUE IMAGES")
print("=" * 60)

# Source directory (same as original notebook)
source_issues_root = RAW_DATASET_DIR / "dataset_issues" / "Skin v2"

# Original mapping (preserved exactly)
issues_mapping = {
    "Dark_Spots": "dark spots",
    "Pigmentation": "dark spots",
    "Pores": "pores",
    "Wrinkles": "wrinkles"
}

summary = {}

for master_label, source_folder_name in issues_mapping.items():

    dest_folder = MASTER_DATASET_DIR / master_label
    dest_folder.mkdir(parents=True, exist_ok=True)

    source_folder = source_issues_root / source_folder_name

    counter = 0

    if source_folder.exists():

        print(f"\nProcessing : {source_folder.relative_to(PROJECT_ROOT)}")

        for file_path in source_folder.glob("*"):

            if file_path.suffix.lower() in [".png", ".jpg", ".jpeg"]:

                new_filename = f"{master_label.lower()}_{counter}{file_path.suffix}"

                shutil.copy(file_path, dest_folder / new_filename)

                counter += 1

    summary[master_label] = counter

    print(f"✓ {master_label:<15}: {counter} images")

# Remove temporary extraction folder (same as original notebook)
temp_extract_folder = RAW_DATASET_DIR / "dataset_issues"

if temp_extract_folder.exists():
    shutil.rmtree(temp_extract_folder)
    print("\n✓ Temporary extraction folder removed.")

print("\n✓ Consolidation completed successfully.\n")

print("Summary")

for label, count in summary.items():
    print(f"{label:<15}: {count}")

print("\nDestination")
print(MASTER_DATASET_DIR)

print("=" * 60)

CONSOLIDATING SKIN ISSUE IMAGES

Processing : datasets\raw\dataset_issues\Skin v2\dark spots
✓ Dark_Spots     : 2126 images

Processing : datasets\raw\dataset_issues\Skin v2\dark spots
✓ Pigmentation   : 2126 images

Processing : datasets\raw\dataset_issues\Skin v2\pores
✓ Pores          : 1632 images

Processing : datasets\raw\dataset_issues\Skin v2\wrinkles
✓ Wrinkles       : 1982 images

✓ Temporary extraction folder removed.

✓ Consolidation completed successfully.

Summary
Dark_Spots     : 2126
Pigmentation   : 2126
Pores          : 1632
Wrinkles       : 1982

Destination
C:\Users\vnman\Desktop\virtual-internship\datasets\master_skin_dataset


In [11]:
# ============================================================
# STEP 11 : Download Dataset 4
# ============================================================

import os
import subprocess
import zipfile

print("=" * 60)
print("DOWNLOADING DATASET 4")
print("=" * 60)

dataset_name = "tiswan14/acne-dataset-image"

zip_file = RAW_DATASET_DIR / "acne-dataset-image.zip"
extract_path_acne = RAW_DATASET_DIR / "dataset_acne_variants"

extract_path_acne.mkdir(parents=True, exist_ok=True)

print(f"Dataset           : {dataset_name}")
print(f"Destination       : {RAW_DATASET_DIR}")

# Download dataset
result = subprocess.run(
    [
        "kaggle",
        "datasets",
        "download",
        "-d",
        dataset_name,
        "-p",
        str(RAW_DATASET_DIR),
    ],
    capture_output=True,
    text=True,
)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Dataset download failed.")

print("\n✓ Download completed successfully.")

print(f"\nDownloaded File : {zip_file.name}")

print("\nExtracting dataset...")

with zipfile.ZipFile(zip_file, "r") as zip_ref:
    zip_ref.extractall(extract_path_acne)

print("✓ Extraction completed successfully.")

# Remove ZIP file
if zip_file.exists():
    zip_file.unlink()

print("✓ ZIP file removed.")

print("\nTop-level contents:")

for item in extract_path_acne.iterdir():
    print(f"• {item.name}")

print("\nFolder Structure Found")

for root, dirs, files in os.walk(extract_path_acne):
    if files:
        image_files = [
            f for f in files
            if f.lower().endswith((".png", ".jpg", ".jpeg"))
        ]

        relative_path = os.path.relpath(root, extract_path_acne)

        print(f"{relative_path} | Images found : {len(image_files)}")

print("\nSummary")
print(f"Extract Location : {extract_path_acne}")

print("=" * 60)

DOWNLOADING DATASET 4
Dataset           : tiswan14/acne-dataset-image
Destination       : C:\Users\vnman\Desktop\virtual-internship\datasets\raw

✓ Download completed successfully.

Downloaded File : acne-dataset-image.zip

Extracting dataset...
✓ Extraction completed successfully.
✓ ZIP file removed.

Top-level contents:
• AcneDataset

Folder Structure Found
AcneDataset\test | Images found : 0
AcneDataset\test\Blackheads | Images found : 265
AcneDataset\test\Cyst | Images found : 189
AcneDataset\test\Papules | Images found : 202
AcneDataset\test\Pustules | Images found : 205
AcneDataset\test\Whiteheads | Images found : 57
AcneDataset\train | Images found : 0
AcneDataset\train\Blackheads | Images found : 735
AcneDataset\train\Cyst | Images found : 645
AcneDataset\train\Papules | Images found : 621
AcneDataset\train\Pustules | Images found : 584
AcneDataset\train\Whiteheads | Images found : 193
AcneDataset\valid | Images found : 0
AcneDataset\valid\Blackheads | Images found : 240
AcneDa

In [12]:
# ============================================================
# STEP 12 : Consolidate Acne Images into Master Dataset
# ============================================================

import shutil

print("=" * 60)
print("CONSOLIDATING ACNE IMAGES")
print("=" * 60)

# Source path (same logic as original notebook)
source_acne_root = RAW_DATASET_DIR / "dataset_acne_variants" / "AcneDataset"

subfolders = ["train", "test", "valid"]

# Destination folders
blackheads_dest = MASTER_DATASET_DIR / "Blackheads"
whiteheads_dest = MASTER_DATASET_DIR / "Whiteheads"
inflammatory_dest = MASTER_DATASET_DIR / "Inflammatory_Acne"

for folder in [blackheads_dest, whiteheads_dest, inflammatory_dest]:
    folder.mkdir(parents=True, exist_ok=True)

# Counters (same as original notebook)
counters = {
    "Blackheads": 0,
    "Whiteheads": 0,
    "Inflammatory": 0
}

for sub in subfolders:

    current_sub_path = source_acne_root / sub

    if not current_sub_path.exists():
        continue

    print(f"\nProcessing : {current_sub_path.relative_to(PROJECT_ROOT)}")

    for folder in current_sub_path.iterdir():

        if not folder.is_dir():
            continue

        category = folder.name

        # Original mapping (preserved exactly)
        if category == "Blackheads":
            target_dest = blackheads_dest
            label_key = "Blackheads"

        elif category == "Whiteheads":
            target_dest = whiteheads_dest
            label_key = "Whiteheads"

        elif category in ["Papules", "Pustules", "Cyst"]:
            target_dest = inflammatory_dest
            label_key = "Inflammatory"

        else:
            continue

        for file_path in folder.glob("*"):

            if file_path.suffix.lower() in [".png", ".jpg", ".jpeg"]:

                new_filename = (
                    f"{label_key.lower()}_"
                    f"{counters[label_key]}"
                    f"{file_path.suffix}"
                )

                shutil.copy(file_path, target_dest / new_filename)

                counters[label_key] += 1

# Remove temporary extraction folder
temp_extract_folder = RAW_DATASET_DIR / "dataset_acne_variants"

if temp_extract_folder.exists():
    shutil.rmtree(temp_extract_folder)
    print("\n✓ Temporary extraction folder removed.")

print("\n✓ Consolidation completed successfully.\n")

print("Summary")
print(f"Blackheads          : {counters['Blackheads']}")
print(f"Whiteheads          : {counters['Whiteheads']}")
print(f"Inflammatory_Acne   : {counters['Inflammatory']}")

print("\nDestination")
print(MASTER_DATASET_DIR)

print("=" * 60)

CONSOLIDATING ACNE IMAGES

Processing : datasets\raw\dataset_acne_variants\AcneDataset\train

Processing : datasets\raw\dataset_acne_variants\AcneDataset\test

Processing : datasets\raw\dataset_acne_variants\AcneDataset\valid

✓ Temporary extraction folder removed.

✓ Consolidation completed successfully.

Summary
Blackheads          : 1240
Whiteheads          : 299
Inflammatory_Acne   : 3078

Destination
C:\Users\vnman\Desktop\virtual-internship\datasets\master_skin_dataset


In [13]:
# ============================================================
# STEP 13 : Verify Master Dataset Distribution
# ============================================================

print("=" * 60)
print("FINAL MASTER DATASET DISTRIBUTION")
print("=" * 60)

total_all = 0

for folder in sorted(MASTER_DATASET_DIR.iterdir()):

    if folder.is_dir():

        count = len([
            f for f in folder.glob("*")
            if f.is_file()
        ])

        print(f"Label : {folder.name:<22} Images : {count}")

        total_all += count

print("-" * 60)
print(f"Grand Total Images : {total_all}")
print("=" * 60)

FINAL MASTER DATASET DISTRIBUTION
Label : Blackheads             Images : 1240
Label : Dark_Spots             Images : 2126
Label : Inflammatory_Acne      Images : 3078
Label : Normal                 Images : 1274
Label : Pigmentation           Images : 2126
Label : Pores                  Images : 1632
Label : Redness                Images : 30
Label : Whiteheads             Images : 299
Label : Wrinkles               Images : 1982
------------------------------------------------------------
Grand Total Images : 13787


In [14]:
# ============================================================
# STEP 14 : Create Balanced Dataset
# ============================================================

import random
import shutil
from PIL import Image

print("=" * 60)
print("CREATING BALANCED DATASET")
print("=" * 60)

# Source and destination directories
source_master = MASTER_DATASET_DIR
balanced_dataset_path = BALANCED_DATASET_DIR
balanced_dataset_path.mkdir(parents=True, exist_ok=True)

TARGET_COUNT = 1200

print(f"Target Images Per Class : {TARGET_COUNT}")
print("\nStarting hybrid balancing process...\n")

for folder in sorted(source_master.iterdir()):

    if not folder.is_dir():
        continue

    class_name = folder.name

    dest_folder = balanced_dataset_path / class_name
    dest_folder.mkdir(parents=True, exist_ok=True)

    # Gather all valid images
    all_images = sorted([
        f for f in folder.glob("*")
        if f.suffix.lower() in [".png", ".jpg", ".jpeg"]
    ])

    raw_count = len(all_images)

    # ---------------------------------------------------------
    # CASE A : Downsample
    # ---------------------------------------------------------
    if raw_count >= TARGET_COUNT:

        selected_images = random.sample(all_images, TARGET_COUNT)

        for idx, img_path in enumerate(selected_images):

            shutil.copy(
                img_path,
                dest_folder / f"{class_name.lower()}_{idx}{img_path.suffix}"
            )

        print(
            f"{class_name:<22}"
            f"Raw : {raw_count:<5}"
            f" -> Downsampled to : {TARGET_COUNT}"
        )

    # ---------------------------------------------------------
    # CASE B : Upsample using augmentation
    # ---------------------------------------------------------
    else:

        # Copy originals
        for idx, img_path in enumerate(all_images):

            shutil.copy(
                img_path,
                dest_folder / f"{class_name.lower()}_{idx}{img_path.suffix}"
            )

        generated_count = raw_count

        while generated_count < TARGET_COUNT:

            for img_path in all_images:

                if generated_count >= TARGET_COUNT:
                    break

                try:

                    with Image.open(img_path) as img:

                        aug_type = random.choice([
                            "flip_h",
                            "flip_v",
                            "rotate_90",
                            "rotate_180",
                            "rotate_270"
                        ])

                        if aug_type == "flip_h":
                            aug_img = img.transpose(Image.FLIP_LEFT_RIGHT)

                        elif aug_type == "flip_v":
                            aug_img = img.transpose(Image.FLIP_TOP_BOTTOM)

                        elif aug_type == "rotate_90":
                            aug_img = img.transpose(Image.ROTATE_90)

                        elif aug_type == "rotate_180":
                            aug_img = img.transpose(Image.ROTATE_180)

                        elif aug_type == "rotate_270":
                            aug_img = img.transpose(Image.ROTATE_270)

                        aug_filename = (
                            f"{class_name.lower()}_aug_{generated_count}.jpg"
                        )

                        aug_img.convert("RGB").save(
                            dest_folder / aug_filename,
                            "JPEG"
                        )

                        generated_count += 1

                except Exception:
                    continue

        print(
            f"{class_name:<22}"
            f"Raw : {raw_count:<5}"
            f" -> Augmented to : {generated_count}"
        )

print("\n" + "=" * 60)
print("VERIFYING BALANCED DATASET")
print("=" * 60)

total_balanced = 0

for folder in sorted(balanced_dataset_path.iterdir()):

    if folder.is_dir():

        count = len([
            f for f in folder.glob("*")
            if f.is_file()
        ])

        print(f"{folder.name:<22} : {count}")

        total_balanced += count

print("-" * 60)
print(f"Grand Total Images : {total_balanced}")
print("=" * 60)

CREATING BALANCED DATASET
Target Images Per Class : 1200

Starting hybrid balancing process...

Blackheads            Raw : 1240  -> Downsampled to : 1200
Dark_Spots            Raw : 2126  -> Downsampled to : 1200
Inflammatory_Acne     Raw : 3078  -> Downsampled to : 1200
Normal                Raw : 1274  -> Downsampled to : 1200
Pigmentation          Raw : 2126  -> Downsampled to : 1200
Pores                 Raw : 1632  -> Downsampled to : 1200
Redness               Raw : 30    -> Augmented to : 1200
Whiteheads            Raw : 299   -> Augmented to : 1200
Wrinkles              Raw : 1982  -> Downsampled to : 1200

VERIFYING BALANCED DATASET
Blackheads             : 1200
Dark_Spots             : 1200
Inflammatory_Acne      : 1200
Normal                 : 1200
Pigmentation           : 1200
Pores                  : 1200
Redness                : 1200
Whiteheads             : 1200
Wrinkles               : 1200
------------------------------------------------------------
Grand Total Images

In [15]:
# ============================================================
# STEP 15 : Create Final Train / Validation / Test Split
# ============================================================

import random
from PIL import Image

print("=" * 60)
print("CREATING FINAL TRAIN / VALIDATION / TEST SPLIT")
print("=" * 60)

# Source dataset (same as original notebook)
source_master = MASTER_DATASET_DIR

# Destination dataset
split_dataset_path = FINAL_SPLIT_DATASET_DIR

splits = ["train", "val", "test"]
classes = [
    folder.name
    for folder in source_master.iterdir()
    if folder.is_dir()
]

# Create folder structure
for split in splits:
    for cls in classes:
        (split_dataset_path / split / cls).mkdir(
            parents=True,
            exist_ok=True
        )

TARGET_TRAIN_COUNT = 840
TARGET_VAL_COUNT = 180
TARGET_TEST_COUNT = 180

print("Target Distribution")
print(f"Train : {TARGET_TRAIN_COUNT}")
print(f"Val   : {TARGET_VAL_COUNT}")
print(f"Test  : {TARGET_TEST_COUNT}")

print("\nStarting preprocessing...\n")


def process_and_save_image(src_path, dest_path, augmentation=None):
    """
    Opens image, converts to RGB,
    resizes to 224x224,
    applies optional augmentation,
    saves as JPEG.
    """

    try:

        with Image.open(src_path) as img:

            img = img.convert("RGB")

            img = img.resize(
                (224, 224),
                Image.Resampling.BILINEAR
            )

            if augmentation == "flip_h":
                img = img.transpose(Image.FLIP_LEFT_RIGHT)

            elif augmentation == "flip_v":
                img = img.transpose(Image.FLIP_TOP_BOTTOM)

            elif augmentation == "rotate_90":
                img = img.transpose(Image.ROTATE_90)

            img.save(
                dest_path,
                "JPEG",
                quality=95
            )

            return True

    except Exception:
        return False


for cls in sorted(classes):

    src_folder = source_master / cls

    all_images = sorted([
        f
        for f in src_folder.glob("*")
        if f.suffix.lower() in [".png", ".jpg", ".jpeg"]
    ])

    raw_count = len(all_images)

    random.seed(42)
    random.shuffle(all_images)

    # Validation and Test use ORIGINAL images only
    val_size = min(
        TARGET_VAL_COUNT,
        max(3, int(raw_count * 0.15))
    )

    test_size = min(
        TARGET_TEST_COUNT,
        max(3, int(raw_count * 0.15))
    )

    val_files = all_images[:val_size]

    test_files = all_images[
        val_size:
        val_size + test_size
    ]

    train_files_raw = all_images[
        val_size + test_size:
    ]

    # -------------------------------
    # Validation
    # -------------------------------

    val_saved = 0

    for idx, f in enumerate(val_files):

        dest = (
            split_dataset_path
            / "val"
            / cls
            / f"val_{idx}.jpg"
        )

        if process_and_save_image(f, dest):
            val_saved += 1

    # -------------------------------
    # Test
    # -------------------------------

    test_saved = 0

    for idx, f in enumerate(test_files):

        dest = (
            split_dataset_path
            / "test"
            / cls
            / f"test_{idx}.jpg"
        )

        if process_and_save_image(f, dest):
            test_saved += 1

    # -------------------------------
    # Train
    # -------------------------------

    train_dest = (
        split_dataset_path
        / "train"
        / cls
    )

    train_saved = 0

    if len(train_files_raw) >= TARGET_TRAIN_COUNT:

        selected_train = random.sample(
            train_files_raw,
            TARGET_TRAIN_COUNT
        )

        for idx, f in enumerate(selected_train):

            dest = train_dest / f"train_{idx}.jpg"

            if process_and_save_image(f, dest):
                train_saved += 1

    else:

        for idx, f in enumerate(train_files_raw):

            dest = train_dest / f"train_{idx}.jpg"

            if process_and_save_image(f, dest):
                train_saved += 1

        if train_saved > 0:

            while train_saved < TARGET_TRAIN_COUNT:

                for f in train_files_raw:

                    if train_saved >= TARGET_TRAIN_COUNT:
                        break

                    aug = random.choice([
                        "flip_h",
                        "flip_v",
                        "rotate_90"
                    ])

                    dest = train_dest / f"train_aug_{train_saved}.jpg"

                    if process_and_save_image(
                        f,
                        dest,
                        augmentation=aug
                    ):
                        train_saved += 1

    print(
        f"{cls:<20}"
        f" Train: {train_saved:<4}"
        f" Val: {val_saved:<4}"
        f" Test: {test_saved:<4}"
    )

print("\n" + "=" * 60)
print("FINAL DATASET CREATED SUCCESSFULLY")
print("=" * 60)

CREATING FINAL TRAIN / VALIDATION / TEST SPLIT
Target Distribution
Train : 840
Val   : 180
Test  : 180

Starting preprocessing...

Blackheads           Train: 840  Val: 180  Test: 180 
Dark_Spots           Train: 840  Val: 180  Test: 180 
Inflammatory_Acne    Train: 840  Val: 180  Test: 180 
Normal               Train: 840  Val: 180  Test: 180 
Pigmentation         Train: 840  Val: 180  Test: 180 
Pores                Train: 840  Val: 180  Test: 180 
Redness              Train: 840  Val: 4    Test: 4   
Whiteheads           Train: 840  Val: 44   Test: 44  
Wrinkles             Train: 840  Val: 180  Test: 180 

FINAL DATASET CREATED SUCCESSFULLY
